In [209]:
import numpy as np
from collections.abc import Callable
import functools
import tqdm

In [210]:
input = open('data/input.txt', 'r').read().split('\n')

In [211]:
N = int(input[0])
hard = np.array(list(map(int, input[1].split())))
time = np.array(list(map(float, input[2].split())))
M = int(input[3])
devs = np.array([list(map(float,i.split())) for i in input[4:]])

In [212]:
rng = np.random.default_rng(seed=22)

In [213]:
def create_subject() -> np.ndarray:
    """
    
    Create subject (solution to one task)

    """
    return rng.integers(0, M, size=N)

In [214]:
def create_population(k: int) -> np.ndarray:
    """
    
    Create k subjects (each subject is solution to one task)
    
    """
    return rng.integers(0, M, size=(k, N))

In [215]:
def fitness_numpy(subject: np.ndarray) -> np.float64:
    """
    
    Quality function for subject
    
    """ 
    task_times = time * devs[subject, hard - 1]
    dev_time = np.bincount(subject, weights=task_times)
    return dev_time.max()

In [216]:
def fitness_numpy_population(population: np.ndarray) -> np.ndarray:
    """
    
    Quality function for population
    
    """ 
    k = population.shape[0]
    task_times = time * devs[population, hard - 1]
    total_time = np.zeros((k, M), dtype=np.float64)
    np.add.at(total_time, (np.arange(k)[:, None], population), task_times)

    return total_time.max(axis=1)

In [217]:
def selection_elitism_and_roulette(fitness: np.ndarray, n: int, elitism_count: int = 5) -> np.ndarray:
    """
    We select the best individuals through elitism and roulette 

    Caution: Individuals selected during Hellenism may be selected again during the roulette

    """
    n = n - elitism_count
    return np.concatenate((fitness.argsort()[:elitism_count], rng.choice(fitness.shape[0], size=n, replace=False)))

In [ ]:
def two_point_crossover(population: np.ndarray, n: int, p: float = 0.7) -> np.ndarray:
    """
    
    Population crossover of n new subjects with probability p

    The better the subject, the higher the probability of gene transmission

    """
    k = population.shape[0]

    parent_left = rng.choice(k, size=n)
    parent_right = (parent_left + rng.integers(k, size=n)) % k

    f_left  = fitness_numpy_population(population[parent_left])
    f_right = fitness_numpy_population(population[parent_right])

    p = np.where(f_left < f_right, p, 1 - p)

    mask = rng.random(size=(n, N)) < p[:, None]

    children = population[parent_left].copy()

    children = np.where(
        mask,
        population[parent_right],
        children
    )
    
    return children

In [ ]:
def mutate(
    children: np.ndarray,
    p_subject_mutation: float = 0.1,
    p_gen_mutation: float = 0.1
) -> np.ndarray:
    """
    
    Random change in an subject's genes

    p_subject_mutation - chance of mutation in population (chance to be selected for mutation)

    p_gen_mutation - chance of gen mutation (the probability of gene change in a selected subject)

    """

    k = children.shape[0]

    mutated = children.copy()

    mask_subject = rng.random(size=k) < p_subject_mutation
    mask_gen = rng.random(size=(k, N)) < p_gen_mutation

    full_mask = mask_subject[:, None] & mask_gen

    random_devs = rng.integers(0, M, size=(k, N))

    mutated = np.where(
        full_mask,
        random_devs,
        mutated
    )

    return mutated

In [220]:
def train(
    create_population: Callable[[], np.ndarray],
    fitness: Callable[[np.ndarray], np.ndarray],
    selection: Callable[[np.ndarray], np.ndarray],
    crossover: Callable[[np.ndarray, int], np.ndarray],
    mutation: Callable[[np.ndarray], np.ndarray],
    n_best: int,
    n_iter: int
) -> np.ndarray:
    population = create_population()

    for _ in tqdm.trange(n_iter):
        f = fitness(population)
        selected = population[selection(f)]
        children = crossover(selected, population.shape[0] - selected.shape[0])
        children = mutation(children)
        population = np.concatenate([selected, children], axis=0)
    
    return population[np.argsort(fitness(population))[:n_best]]

In [221]:
solutions = train(
        create_population=functools.partial(create_population, 100),
        fitness=fitness_numpy_population,
        selection=functools.partial(selection_elitism_and_roulette, n=25),
        crossover=two_point_crossover,
        mutation=functools.partial(mutate, p_subject_mutation=0.001, p_gen_mutation=0.5),
        n_best=1,
        n_iter=5000
    )

100%|██████████| 5000/5000 [00:51<00:00, 96.21it/s] 


In [ ]:
solutions = solutions.reshape(N)
print(fitness_numpy(solutions))
solutions = solutions+1

res = " ".join(map(str, solutions))
open('data/output.txt','w').write(res)

605.7599999999998
